In [1]:
import sys
sys.path.append('./')  # Adjust the path if necessary

from data_loader import load_user_reviews
from model_pipeline import RecommenderModel
from utils import extract_latest_n_reviews, extract_product_names
from retrieval import initialize_chromadb, get_or_create_collection, collect_results_alternating_shortest
from evaluation import normalize, compute_similarity, recall_at_k, ndcg_at_k

# Additional imports
import numpy as np
import torch
import pandas as pd

# Initialize Recommender Model
recommender_model = RecommenderModel()

# Initialize ChromaDB
db_path = "./chroma_db_mpnet"
db = initialize_chromadb(db_path)
collection_name = 'product_embeddings'  # Use the same collection name
collection = get_or_create_collection(db, collection_name)

# Load the product data
df = pd.read_csv("data/meta_all_beauty_filtered_simple.csv")

# Load training data
train_file = 'data/train_val_user_reviews.json'
input_set = load_user_reviews(train_file)

# Load test data
test_file = 'data/test_user_reviews.json'
input_set_test = load_user_reviews(test_file)


c:\Users\Trung\anaconda3\envs\master_torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


max length is 2048


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


models/hf-frompretrained-download/meta-llama/Meta-Llama-3-8B-Instruct


`low_cpu_mem_usage` was None, now set to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]
⚠️ It looks like you upgraded from a version below 0.6 and could benefit from vacuuming your database. Run chromadb utils vacuum --help for more information.


In [2]:
input_set_test[0]

{'user_id': 'AFSKPY37N3C43SOI5IEXEK5JSIYA',
 'reviews': [{'product_name': 'NIRA Skincare Laser & Serum Bundle - Includes Anti-Aging Laser & Hyaluronic Acid Serum - Reduces Appearance of Fine Lines & Wrinkles - FDA Cleared',
   'parent_asin': 'B08P2DZB4X',
   'rating': 5.0,
   'title': 'Great for at home use and so easy to use!',
   'text': 'This is perfect for my between salon visits. I have been using this now twice a week for over a month and I absolutely love it! My skin looks amazing and feels super smooth and silky. This is also super easy to use (just follow instructions). I can see already that I will begin expanding the time between visits which will definitely help me save money in the long run. Highly recommend!',
   'timestamp': 1627391044559}]}

In [4]:
# Parameters
n_latest_reviews = 3
SIMILARITY_THRESHOLD = 90.0

# Select a specific user index
userIndex = 0  # Change this index to select a different user

# Start processing for a single user
print(f"Processing user {userIndex + 1}")

# Extract latest reviews for training
example_user = [input_set[userIndex]]
latest_reviews = extract_latest_n_reviews(example_user, n_latest_reviews)
review_text = "\n".join([rev['text'] for rev in latest_reviews])

# Display the user's latest reviews
print("\nUser's Latest Reviews:")
for idx, rev in enumerate(latest_reviews, 1):
    print(f"{idx}. {rev['text']}")

# Generate the prompt for user profile
from config import USER_PROFILE_PROMPT

user_profile_prompt = USER_PROFILE_PROMPT.format(reviews=review_text)
print("\nUser Profile Generation Prompt:")
print(user_profile_prompt)

# Create user profile
profile = recommender_model.create_user_profile(review_text)
print("\nGenerated User Profile:")
print(profile)

# Generate the prompt for preliminary recommendations
from config import PRELIMINARY_RECOMMENDATIONS_PROMPT

prelim_rec_prompt = PRELIMINARY_RECOMMENDATIONS_PROMPT.format(user_profile=profile)
print("\nPreliminary Recommendations Generation Prompt:")
print(prelim_rec_prompt)

# Generate preliminary recommendations
preliminary_rec = recommender_model.create_preliminary_recommendations(profile)
print("\nPreliminary Recommendations:")
print(preliminary_rec)

# Extract product names
product_names = extract_product_names(preliminary_rec)
print("\nExtracted Product Names:")
for idx, name in enumerate(product_names, 1):
    print(f"{idx}. {name}")

# Check if product_names is empty
if not product_names:
    print("No product names extracted. Exiting.")
else:
    # Query ChromaDB and collect results
    final_results = collect_results_alternating_shortest(product_names, collection)

    # Check if collect_results_alternating_shortest returned -1
    if final_results == -1:
        print("No recommendations found. Exiting.")
    else:
        # Proceed with evaluation if we have recommendations
        # Get the actual product from the test set
        example_user_test = [input_set_test[userIndex]]
        test_review = extract_latest_n_reviews(example_user_test, 1)
        test_product = test_review[0]['parent_asin']
        print("\nTest Product:")
        print(test_product)

        # Retrieve recommended products
        recommended_products = []
        for doc, metadata in final_results:
            # Retrieve the product title using metadata (e.g., 'asin')
            #print(final_results)
            asin = metadata['metadata']
            filt = df['parent_asin'] == asin
            title = asin
            print(asin)
            if len(title) > 0:
                recommended_products.append(title[0])
            else:
                recommended_products.append(doc)  # Fallback to the document if title not found

        # Display recommended products with metadata
        print("\nRecommended Products:")
        for idx, (prod, (doc, metadata)) in enumerate(zip(recommended_products, final_results), 1):
            print(f"{idx}. {prod}")
            print(f"   Document: {doc}")
            print(f"   Metadata: {metadata}")

        # Evaluate recommendations
        normalized_test_product = normalize(test_product)
        normalized_recommended_products = [normalize(name) for name in recommended_products]

        # Compute similarity scores and matches
        similarity_scores = []
        matches = []
        for rec_product in normalized_recommended_products:
            sim_score = compute_similarity(rec_product, normalized_test_product)
            similarity_scores.append(sim_score)
            matches.append(sim_score >= SIMILARITY_THRESHOLD)

        # Display similarity scores and matches
        print("\nSimilarity Scores and Matches:")
        for idx, (prod, score, match) in enumerate(zip(recommended_products, similarity_scores, matches), 1):
            print(f"{idx}. {prod}")
            print(f"   Similarity Score: {score:.2f}%")
            print(f"   Match: {'Yes' if match else 'No'}")

        # Calculate Recall@K and NDCG@K
        from evaluation import recall_at_k, ndcg_at_k

        K = 10  # For Recall@K and NDCG@K
        recall = recall_at_k(matches, K)
        ndcg = ndcg_at_k(matches, K)

        # Display evaluation metrics
        print("\nEvaluation Metrics:")
        print(f"Recall@{K}: {recall}")
        print(f"NDCG@{K}: {ndcg}")

Processing user 1

User's Latest Reviews:
1. Really nice small brush. Made well, nice wood made with boar bristle, my son absolutely loves this. It brushes his hair well and keeps him looking his best. This compact size makes it nice to keep in the center console of his car or to take on vacation with him. Highly recommend!
2. I try to get Keratin treatments every 3 months, but honestly it has been getting costly. So, when I saw this I was excited to try it. I found it difficult to use and almost impossible to get to saturate the back of my hair and straight iron it the way they do in the salon. Front and sides were ok, but I couldn't maneuver the back to get it straight. Then I saw the ingredients after the first time and saw it contained formaldehyde and that was the last time I used the actual treatment. I did, however, use the shampoo and conditioner (and I still am). I wish they sold the S&C separate because I really did like it and I am always in the market for a good hair wash w

Add of existing embedding ID: 670f8d76324e0602c00cb6ce
Add of existing embedding ID: 670f8d76324e0602c00cb6cf
Add of existing embedding ID: 670f8d76324e0602c00cb6d0
Add of existing embedding ID: 670f8d76324e0602c00cb6d1
Add of existing embedding ID: 670f8d76324e0602c00cb6d2
Add of existing embedding ID: 670f8d76324e0602c00cb6d3
Add of existing embedding ID: 670f8d76324e0602c00cb6d4
Add of existing embedding ID: 670f8d76324e0602c00cb6d5
Add of existing embedding ID: 670f8d76324e0602c00cb6d6
Add of existing embedding ID: 670f8d76324e0602c00cb6d7
Add of existing embedding ID: 670f8d76324e0602c00cb6d8
Add of existing embedding ID: 670f8d76324e0602c00cb6d9
Add of existing embedding ID: 670f8d76324e0602c00cb6da
Add of existing embedding ID: 670f8d76324e0602c00cb6db
Add of existing embedding ID: 670f8d76324e0602c00cb6dc
Add of existing embedding ID: 670f8d76324e0602c00cb6dd
Add of existing embedding ID: 670f8d76324e0602c00cb6de
Add of existing embedding ID: 670f8d76324e0602c00cb6df
Add of exi


Test Product:
B08P2DZB4X
B07H8TY6BJ
B00PL4M75Q
B018LP2SMS

Recommended Products:
1. B
   Document: B07H8TY6BJ
   Metadata: {'metadata': 'B07H8TY6BJ'}
2. B
   Document: B00PL4M75Q
   Metadata: {'metadata': 'B00PL4M75Q'}
3. B
   Document: B018LP2SMS
   Metadata: {'metadata': 'B018LP2SMS'}

Similarity Scores and Matches:
1. B
   Similarity Score: 18.18%
   Match: No
2. B
   Similarity Score: 18.18%
   Match: No
3. B
   Similarity Score: 18.18%
   Match: No

Evaluation Metrics:
Recall@10: 0.0
NDCG@10: 0.0
